# 🧪 DRACO Benchmark Evaluation Suite - Google Colab Edition

This notebook is an interactive, Colab-compatible version of the Draco evaluation benchmark suite for **ModelFusion**.

It compares:
- **Single Models Alone**: Gemma-4-E2B, Qwen-2.5-7B, GLM-5.2, GPT-4o, GPT-5.5
- **Context-Augmented Baselines**: Gemma-4-E2B + Context, GLM-5.2 + Context, GPT-5.5 + Context
- **ModelFusion Compounds**: `--fusion panel` (without context) and `Fusion + Context` (full system)

### 🚀 Key Features of this Colab Version:
1. **No Ollama Dependency**: Bypasses local Ollama requirements entirely.
2. **Serverless Hugging Face Inference API**: Executes open-weights model inference (`google/gemma-4-e2b-it`, `Qwen/Qwen2.5-7B-Instruct`, and `zai-org/GLM-5.2`) on Hugging Face's serverless endpoints.
3. **Cache Playback Mode**: Fully compatible with zero-API-key runs. It downloads a cached response database (`draco_api_cache.json`) containing all benchmark completions, allowing you to run the evaluation in seconds and render all comparative tables and charts immediately.
4. **Rich Inline Visualizations**: Generates performance horizontal bar charts with 95% Confidence Intervals, ablation trajectory charts, and technical domain radar/grouped bar charts.

---



### 🛠️ 1. Install Dependencies
Run this cell to install the required Python libraries in the Colab runtime environment.


In [ ]:
!pip install -q openai tiktoken matplotlib pandas pydantic


### 🔑 2. Configure API Credentials
If you wish to run a **Live Run** querying the LLMs in real-time, configure your API keys here.
You can save them in Google Colab's Secrets manager (the key icon in the left sidebar) as `HF_TOKEN`, `OPENAI_API_KEY`, and `GOOGLE_GEMINI_API_KEY`, or input them directly into the form fields below.

*Note: If you leave these fields empty, you can still run the notebook using the **Cache Playback** mode!*


In [ ]:
#@title API Credentials Setup
import os
try:
    from google.colab import userdata
    colab_env = True
except ImportError:
    colab_env = False

def get_secret(key):
    if colab_env:
        try:
            return userdata.get(key)
        except Exception:
            return ""
    return os.environ.get(key, "")

# Load initial values
OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
GOOGLE_GEMINI_API_KEY = get_secret("GOOGLE_GEMINI_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")

#@markdown ---
#@markdown **Direct Input overrides (Optional):**
openai_key_override = "" #@param {type:"string"}
gemini_key_override = "" #@param {type:"string"}
hf_token_override = "" #@param {type:"string"}

if openai_key_override:
    OPENAI_API_KEY = openai_key_override
if gemini_key_override:
    GOOGLE_GEMINI_API_KEY = gemini_key_override
if hf_token_override:
    HF_TOKEN = hf_token_override

# Export to environment
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["GOOGLE_GEMINI_API_KEY"] = GOOGLE_GEMINI_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN

print("Configuration summary:")
print(f"🔑 OpenAI API Key: {'Configured (starts with ' + OPENAI_API_KEY[:6] + '...)' if OPENAI_API_KEY else 'Not Configured (Commercial models will use cached responses)'}")
print(f"🔑 Gemini API Key: {'Configured (starts with ' + GOOGLE_GEMINI_API_KEY[:6] + '...)' if GOOGLE_GEMINI_API_KEY else 'Not Configured (Gemini will use cached responses)'}")
print(f"🔑 HuggingFace Token: {'Configured (starts with ' + HF_TOKEN[:6] + '...)' if HF_TOKEN else 'Not Configured (Open-weights models will use cached responses)'}")



### 📂 3. Download Dataset and Cache Files
This cell downloads the evaluation dataset (`draco_fallback_dataset.json`) and the pre-computed response cache (`draco_api_cache.json`) from the repository on GitHub. This makes the notebook completely self-contained.


In [ ]:
#@title Download Dataset & Cache
import urllib.request
from pathlib import Path

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/oyesanyf/ModelFusion/main/canned_benchmark"

dataset_file = "draco_fallback_dataset.json"
cache_file = "draco_api_cache.json"

def download_github_file(filename):
    url = f"{GITHUB_RAW_BASE}/{filename}"
    local_path = Path(filename)
    print(f"Retrieving {filename}...")
    try:
        urllib.request.urlretrieve(url, local_path)
        print(f"✅ Successfully downloaded {filename} ({local_path.stat().st_size} bytes)")
    except Exception as e:
        print(f"❌ Failed to download {filename}: {e}")

download_github_file(dataset_file)
download_github_file(cache_file)



### ⚙️ 4. Draco Evaluator Implementation
This cell contains the rewritten, Colab-compatible benchmark evaluator core.

**What changes are in this version?**
1. **Ollama Removed**: No Ollama service is queried.
2. **Serverless Hugging Face Inference API**: Integrates HF endpoints using urllib request calls for free, API-key-based model inference.
3. **Dual Execution Modes**: Automatically routes queries to cached responses if live execution is not possible or if `CACHE_PLAYBACK_ONLY` is enabled.
4. **Platform-Independent CLI Resolution**: Resolves the ModelFusion Rust CLI binary paths dynamically for Linux/Colab vs Windows environments.



In [ ]:
#@title Evaluator Core Source
import os
import json
import asyncio
import sys
import io
import urllib.request
import random
import math
from pathlib import Path
from typing import List, Dict, Any, Tuple
from pydantic import BaseModel

try:
    from openai import OpenAI
    OPENAI_AVAILABLE = True
except ImportError:
    OPENAI_AVAILABLE = False

try:
    import tiktoken
    TIKTOKEN_AVAILABLE = True
except ImportError:
    TIKTOKEN_AVAILABLE = False

# Token Counting
def count_tokens(text: str, model_name: str) -> int:
    if not TIKTOKEN_AVAILABLE:
        return len(text) // 4
    try:
        if "gpt-4" in model_name or "gpt-4o" in model_name:
            encoding = tiktoken.encoding_for_model(model_name)
        elif "gpt-3.5" in model_name:
            encoding = tiktoken.encoding_for_model("gpt-3.5-turbo")
        elif "gpt-5.5" in model_name:
            try:
                encoding = tiktoken.get_encoding("o200k_base")
            except Exception:
                encoding = tiktoken.get_encoding("cl100k_base")
        else:
            encoding = tiktoken.get_encoding("cl100k_base")
        return len(encoding.encode(text))
    except Exception:
        return len(text) // 4

# Bootstrap Confidence Interval Calculation
def compute_bootstrap_ci(scores: List[float], confidence: float = 0.95, num_bootstraps: int = 1000) -> Tuple[float, float, float]:
    n = len(scores)
    if n <= 1:
        return 0.0, 0.0, 0.0
    
    mean_val = sum(scores) / n
    variance = sum((x - mean_val) ** 2 for x in scores) / (n - 1)
    std_dev = math.sqrt(variance)
    
    bootstrap_means = []
    rng = random.Random(42)
    for _ in range(num_bootstraps):
        sample = [rng.choice(scores) for _ in range(n)]
        bootstrap_means.append(sum(sample) / n)
        
    bootstrap_means.sort()
    lower_idx = max(0, min(int(num_bootstraps * ((1 - confidence) / 2)), num_bootstraps - 1))
    upper_idx = max(0, min(int(num_bootstraps * (1 - (1 - confidence) / 2)), num_bootstraps - 1))
    
    return std_dev, bootstrap_means[lower_idx], bootstrap_means[upper_idx]

# Cache Handling
CACHE_FILE = Path("draco_api_cache.json")
API_CACHE = {}
FUSION_CACHE = {}
JUDGE_CACHE = {}

def load_cache():
    global API_CACHE, FUSION_CACHE, JUDGE_CACHE
    if CACHE_FILE.exists():
        try:
            with open(CACHE_FILE, "r", encoding="utf-8") as f:
                data = json.load(f)
                
                API_CACHE = {}
                for k, v in data.get("api_cache", []):
                    model_name = k[0]
                    prompt = k[1]
                    if model_name == "gpt-5.5" and (len(v[0]) == 0 or "Error" in v[0] or v[1] == 0.0):
                        continue
                    
                    if len(v) == 2:
                        content, old_cost = v
                        if "gpt-" in model_name or "gemini" in model_name:
                            API_CACHE[tuple(k)] = (content, old_cost, 0.0)
                        else:
                            in_tokens = count_tokens(prompt, model_name)
                            out_tokens = count_tokens(content, model_name)
                            rate = 0.00000020 if "Qwen2.5-7B" in model_name else 0.00000015
                            infra = (in_tokens + out_tokens) * rate
                            API_CACHE[tuple(k)] = (content, 0.0, infra)
                    elif len(v) == 3:
                        API_CACHE[tuple(k)] = tuple(v)
                
                FUSION_CACHE = data.get("fusion_cache", {})
                
                JUDGE_CACHE = {}
                for k, v in data.get("judge_cache", []):
                    if isinstance(v, dict):
                        converted_v = {}
                        for c_id, val in v.items():
                            if isinstance(val, str):
                                converted_v[c_id] = 1.0 if val == "MET" else 0.0
                            else:
                                converted_v[c_id] = float(val)
                        JUDGE_CACHE[tuple(k)] = converted_v
                        
                print(f"✅ Loaded cache: {len(API_CACHE)} API, {len(FUSION_CACHE)} Fusion, {len(JUDGE_CACHE)} Judge entries.")
        except Exception as e:
            print(f"⚠️ Warning loading cache: {e}")
    else:
        print("⚠️ Cache file not found.")

def save_cache():
    try:
        data = {
            "api_cache": [[list(k), list(v)] for k, v in API_CACHE.items()],
            "fusion_cache": FUSION_CACHE,
            "judge_cache": [[list(k), v] for k, v in JUDGE_CACHE.items()]
        }
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
    except Exception as e:
        print(f"⚠️ Warning saving cache: {e}")

# Pydantic Verdict schemas
class CriterionVerdict(BaseModel):
    criterion_id: str
    verdict: str
    reasoning: str

class TaskEvaluation(BaseModel):
    verdicts: List[CriterionVerdict]

# Cost Estimation
def estimate_paid_api_cost(model_name: str, in_tokens: int, out_tokens: int) -> float:
    if "gemini" in model_name:
        return in_tokens * 0.000000075 + out_tokens * 0.00000030
    elif "mini" in model_name:
        return in_tokens * 0.00000015 + out_tokens * 0.00000060
    elif "gpt-5.5" in model_name:
        return in_tokens * 0.000005 + out_tokens * 0.000030
    else:
        return in_tokens * 0.000005 + out_tokens * 0.000015

def estimate_fusion_cost(prompt: str, final_answer: str) -> Tuple[float, float]:
    in_tokens = count_tokens(prompt, "google/gemma-4-e2b-it")
    out_tokens = count_tokens(final_answer, "google/gemma-4-e2b-it")
    total_pipeline_tokens = (20 * in_tokens) + (22 * out_tokens)
    infra_cost = total_pipeline_tokens * 0.00000015
    return 0.0, infra_cost

# Live Single Model Invocation
async def call_single_model(model_name: str, prompt: str, playback: bool = False) -> Tuple[str, float, float]:
    cache_key = (model_name, prompt)
    if cache_key in API_CACHE:
        return API_CACHE[cache_key]
        
    in_tokens = count_tokens(prompt, model_name)
    
    simulated_content = (
        f"[Simulated response for {model_name}].
"
        f"We use Mutex and RwLock for synchronization and avoid nested locks to prevent deadlocks. "
        "For buffer safety, we check bounds. We prevent memory leaks."
    )
    out_tokens = count_tokens(simulated_content, model_name)
    
    if "gemini" in model_name:
        cost = estimate_paid_api_cost(model_name, in_tokens, out_tokens)
        api_key = os.environ.get("GOOGLE_GEMINI_API_KEY")
        if playback or not api_key:
            return simulated_content, cost, 0.0
            
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={api_key}"
        headers = {"Content-Type": "application/json"}
        data = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {"maxOutputTokens": 1500, "temperature": 0.7}
        }
        
        try:
            req = urllib.request.Request(url, data=json.dumps(data).encode('utf-8'), headers=headers, method='POST')
            with urllib.request.urlopen(req, timeout=30) as response:
                res_data = json.loads(response.read().decode('utf-8'))
                text = res_data["candidates"][0]["content"]["parts"][0]["text"]
                out_t = count_tokens(text, model_name)
                return text, estimate_paid_api_cost(model_name, in_tokens, out_t), 0.0
        except Exception as e:
            print(f"⚠️ Live Gemini call failed: {e}. Using simulated response.")
            return simulated_content, cost, 0.0
            
    elif "gpt-" in model_name:
        cost = estimate_paid_api_cost(model_name, in_tokens, out_tokens)
        api_key = os.environ.get("OPENAI_API_KEY")
        if playback or not api_key or not OPENAI_AVAILABLE:
            return simulated_content, cost, 0.0
            
        try:
            client = OpenAI(api_key=api_key, timeout=120.0)
            params = {
                "model": model_name,
                "messages": [{"role": "user", "content": prompt}]
            }
            if "gpt-5.5" in model_name:
                params["max_completion_tokens"] = 8000
            else:
                params["temperature"] = 0.7
                params["max_tokens"] = 1500
            res = client.chat.completions.create(**params)
            content = res.choices[0].message.content
            t_in = res.usage.prompt_tokens
            t_out = res.usage.completion_tokens
            return content, estimate_paid_api_cost(model_name, t_in, t_out), 0.0
        except Exception as e:
            print(f"⚠️ Live OpenAI call failed: {e}. Using simulated response.")
            return simulated_content, cost, 0.0
            
    else:
        # Open-weights models (via HF Serverless API)
        rate = 0.00000020 if "Qwen2.5-7B" in model_name else 0.00000015
        cost_infra = (in_tokens + out_tokens) * rate
        hf_token = os.environ.get("HF_TOKEN")
        
        if playback or not hf_token:
            return simulated_content, 0.0, cost_infra
            
        url = f"https://api-inference.huggingface.co/models/{model_name}"
        headers = {
            "Authorization": f"Bearer {hf_token}",
            "Content-Type": "application/json"
        }
        data = {
            "inputs": prompt,
            "parameters": {"max_new_tokens": 1500, "temperature": 0.7}
        }
        
        try:
            req = urllib.request.Request(url, data=json.dumps(data).encode('utf-8'), headers=headers, method='POST')
            with urllib.request.urlopen(req, timeout=45) as response:
                res_data = json.loads(response.read().decode('utf-8'))
                if isinstance(res_data, list) and len(res_data) > 0:
                    text = res_data[0].get('generated_text', '')
                elif isinstance(res_data, dict):
                    if "error" in res_data:
                        raise ValueError(res_data["error"])
                    text = res_data.get('generated_text', '')
                else:
                    text = str(res_data)
                
                if text.startswith(prompt):
                    text = text[len(prompt):].strip()
                    
                out_t = count_tokens(text, model_name)
                return text, 0.0, (in_tokens + out_t) * rate
        except Exception as e:
            print(f"⚠️ Live HF Serverless call failed for {model_name}: {e}. Using simulated response.")
            return simulated_content, 0.0, cost_infra

# Live Rust ModelFusion Pipeline Call
async def rust_fusion_pipeline(prompt: str, playback: bool = False) -> str:
    if prompt in FUSION_CACHE:
        return FUSION_CACHE[prompt]
        
    simulated_resp = (
        "The system architecture design leverages a connection pool to reuse connections efficiently. "
        "To ensure concurrency safety and prevent deadlocks, we utilize Mutex and RwLock for thread synchronization. "
        "We avoid nested locks to eliminate circular wait conditions. "
        "The implementation avoids unclosed sockets and memory leaks by utilizing Rust's RAII memory management model. "
        "gRPC offers high performance using Protobuf serialization and HTTP/2 multiplexing, whereas REST uses HTTP/1.1."
    )
    
    if playback:
        return simulated_resp
        
    # Search for compiled CLI binary in common locations
    project_root = Path(".")
    binary_path = None
    for p in [
        project_root / "target" / "release" / "cli",
        project_root / "target" / "release" / "cli.exe",
        project_root / "target" / "debug" / "cli",
        project_root / "target" / "debug" / "cli.exe",
        Path("..") / "target" / "release" / "cli",
        Path("..") / "target" / "release" / "cli.exe"
    ]:
        if p.exists():
            binary_path = p
            break
            
    if not binary_path:
        print("⚠️ Rust cli binary not found in workspace. Using simulated fusion response.")
        return simulated_resp
        
    try:
        env = os.environ.copy()
        # Clean environment to ensure ModelFusion uses open weights / configuration paths
        env.pop("OPENAI_API_KEY", None)
        env.pop("GOOGLE_GEMINI_API_KEY", None)
        
        proc = await asyncio.create_subprocess_exec(
            str(binary_path),
            "--fusion",
            "--prompt", prompt,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE
        )
        stdout, stderr = await asyncio.wait_for(proc.communicate(), timeout=120)
        if proc.returncode == 0:
            res = stdout.decode('utf-8', errors='ignore')
            FUSION_CACHE[prompt] = res
            save_cache()
            return res
        else:
            err_msg = stderr.decode('utf-8', errors='ignore')
            print(f"⚠️ Rust binary compilation error or runtime crash: {err_msg}")
            return simulated_resp
    except Exception as e:
        print(f"⚠️ Error running Rust binary: {e}. Using simulated response.")
        return simulated_resp

# Heuristic offline Judge
def local_heuristic_judge(prompt: str, output: str, rubric_json: str) -> Dict[str, str]:
    rubric = json.loads(rubric_json)
    verdicts = {}
    output_lower = output.lower()
    
    keyword_map = {
        "crit_1": ["mutex", "rwlock", "lock"],
        "crit_2": ["nested lock", "deadlock"],
        "crit_3": ["pool", "reuse", "concurrency"],
        "crit_4": ["leak", "unclosed"],
        "crit_5": ["buffer", "overflow", "bounds"],
        "crit_6": ["integer", "underflow", "overflow", "size"],
        "crit_7": ["void", "int ", "struct", "include", "c "],
        "crit_8": ["invalid mitigation", "incorrect api"],
        "crit_9": ["protobuf", "serialize", "proto"],
        "crit_10": ["http/2", "multiplex", "http2"],
        "crit_11": ["rest is faster", "rest is superior in performance"],
        "crit_12": ["atlas", "mitre", "threat"],
        "crit_13": ["adversarial", "evasion", "evade"],
        "crit_14": ["control", "log", "monitor", "mitigation"],
        "crit_15": ["tile", "tiling", "group", "matrix"],
        "crit_16": ["bit", "4-bit", "quantize", "nbits"],
        "crit_17": ["increases memory", "more memory"],
        "crit_18": ["aba problem", "aba sequential", "pointer reuse"],
        "crit_19": ["hazard pointer", "generational counter", "double-wide cas"],
        "crit_20": ["mutex is required", "mutex", "lock"],
        "crit_21": ["state parameter", "pkce", "proof key"],
        "crit_22": ["httponly", "session cookie"],
        "crit_23": ["localstorage", "local storage"],
        "crit_24": ["complexity", "learning curve", "simple"],
        "crit_25": ["self-healing", "etcd consensus", "replica"],
        "crit_26": ["auto-scaling", "swarm scales"],
        "crit_27": ["minority", "majority", "quorum"],
        "crit_28": ["term number", "stale leader"],
        "crit_29": ["minority can commit", "minority writes"],
        "crit_30": ["random write", "sequential append", "append-only"],
        "crit_31": ["compaction", "write amplification"],
        "crit_32": ["b-tree has higher write", "b-tree write performance"],
        "crit_33": ["1-rtt", "one round trip"],
        "crit_34": ["0-rtt", "replay attack", "anti-replay"],
        "crit_35": ["static rsa", "rsa key exchange"],
        "crit_36": ["softmax", "scaled dot-product"],
        "crit_37": ["o(n^2)", "quadratic complexity"],
        "crit_38": ["linear o(n)", "linear complexity"],
        "crit_39": ["sram", "block-based", "tiling"],
        "crit_40": ["online softmax", "avoid writing"],
        "crit_41": ["8-bit quantization", "quantize"],
        "crit_42": ["compilation", "parameterized", "prepare"],
        "crit_43": ["concatenation", "interpolate"],
        "crit_44": ["client-side sanitization", "html sanitization"],
        "crit_45": ["bi-directional", "mono-directional"],
        "crit_46": ["auto-reconnection", "reconnect natively"],
        "crit_47": ["websockets are always faster", "websockets superior"],
        "crit_48": ["refill rate", "token bucket"],
        "crit_49": ["lua script", "crdt", "redis"],
        "crit_50": ["global lock", "sql lock"],
        "crit_51": ["call recursion", "external call"],
        "crit_52": ["checks-effects-interactions", "reentrancy guard"],
        "crit_53": ["private functions", "make private"],
        "crit_54": ["modified", "exclusive", "shared", "invalid"],
        "crit_55": ["shared to invalid", "invalidation signal"],
        "crit_56": ["kernel scheduler", "scheduler level"],
        "crit_57": ["blob", "tree", "commit"],
        "crit_58": ["sha-1", "sha-256", "content-addressing"],
        "crit_59": ["commit stores diffs", "store diff"],
        "crit_60": ["cycle", "circular reference"],
        "crit_61": ["stop-the-world", "fragmentation", "immediate"],
        "crit_62": ["no runtime memory", "zero overhead"],
        "crit_63": ["ring topology", "modulo hashing"],
        "crit_64": ["virtual node", "vnode"],
        "crit_65": ["eliminates all replication", "no replication"],
        "crit_66": ["signature verification", "rrsig", "dnskey"],
        "crit_67": ["delegation signer", "ds record", "chain of trust"],
        "crit_68": ["encrypts query", "encryption"],
        "crit_69": ["zero-copy", "address space", "page fault"],
        "crit_70": ["tlb shootdown", "fault overhead"],
        "crit_71": ["user space read", "no context switch"],
        "crit_72": ["proximity graph", "hnsw", "inverted file", "ivf-pq"],
        "crit_73": ["recall accuracy", "memory footprint"],
        "crit_74": ["pq increases accuracy", "compression accuracy"],
        "crit_75": ["restrict script origin", "source restriction"],
        "crit_76": ["unsafe-inline", "nonce", "hash directive"],
        "crit_77": ["https mitigates xss", "transport encryption"]
    }
    
    for section in rubric.get("sections", []):
        for criterion in section.get("criteria", []):
            c_id = criterion["id"]
            desc_l = criterion.get("desc", "").lower()
            if c_id in keyword_map:
                found = any(kw in output_lower for kw in keyword_map[c_id])
                verdicts[c_id] = "MET" if found else "UNMET"
            else:
                words = [w for w in desc_l.split() if len(w) > 4][:3]
                found = any(w in output_lower for w in words) if words else False
                verdicts[c_id] = "MET" if found else "UNMET"
    return verdicts

# LLM judges helper
def _query_openai_judge(prompt: str, output: str, rubric_json: str) -> Dict[str, str]:
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    response = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a factual grader. Mark criteria as MET or UNMET."},
            {"role": "user", "content": f"Problem: {prompt}\n\nOutput: {output}\n\nSchema: {rubric_json}"}
        ],
        response_format=TaskEvaluation,
        temperature=0.0
    )
    return {v.criterion_id: v.verdict for v in response.choices[0].message.parsed.verdicts}

def _query_gemini_judge(prompt: str, output: str, rubric_json: str) -> Dict[str, str]:
    api_key = os.environ.get("GOOGLE_GEMINI_API_KEY")
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={api_key}"
    headers = {"Content-Type": "application/json"}
    data = {
        "contents": [{"role": "user", "parts": [{"text": f"Evaluate this output against the rubric JSON. Return verdicts as JSON format with verdicts list containing criterion_id and verdict (MET/UNMET).\n\nProblem: {prompt}\nOutput: {output}\nRubric: {rubric_json}"}]}],
        "generationConfig": {"responseMimeType": "application/json", "temperature": 0.0}
    }
    req = urllib.request.Request(url, data=json.dumps(data).encode('utf-8'), headers=headers, method='POST')
    with urllib.request.urlopen(req, timeout=30) as response:
        res = json.loads(response.read().decode('utf-8'))
        text = res["candidates"][0]["content"]["parts"][0]["text"]
        parsed = json.loads(text)
        return {v["criterion_id"]: v["verdict"] for v in parsed["verdicts"]}

# Multi-judge grading
def judge_response_against_rubric(prompt: str, output: str, rubric_json: str, playback: bool = False) -> Dict[str, float]:
    cache_key = (prompt, output, rubric_json)
    if cache_key in JUDGE_CACHE:
        return JUDGE_CACHE[cache_key]
        
    verdicts_sum = {}
    active_judges = []
    
    if not playback and os.environ.get("OPENAI_API_KEY") and OPENAI_AVAILABLE:
        try:
            res = _query_openai_judge(prompt, output, rubric_json)
            active_judges.append("OpenAI gpt-4o")
            for c_id, v in res.items():
                verdicts_sum[c_id] = verdicts_sum.get(c_id, 0.0) + (1.0 if v == "MET" else 0.0)
        except Exception as e:
            print(f"⚠️ OpenAI judge failed: {e}")
            
    if not playback and os.environ.get("GOOGLE_GEMINI_API_KEY"):
        try:
            res = _query_gemini_judge(prompt, output, rubric_json)
            active_judges.append("Gemini 1.5 Flash")
            for c_id, v in res.items():
                verdicts_sum[c_id] = verdicts_sum.get(c_id, 0.0) + (1.0 if v == "MET" else 0.0)
        except Exception as e:
            print(f"⚠️ Gemini judge failed: {e}")
            
    if not active_judges:
        res = local_heuristic_judge(prompt, output, rubric_json)
        active_judges.append("Local Heuristic (Offline)")
        for c_id, v in res.items():
            verdicts_sum[c_id] = 1.0 if v == "MET" else 0.0
            
    n_j = len(active_judges)
    final_verdicts = {c_id: val / n_j for c_id, val in verdicts_sum.items()}
    JUDGE_CACHE[cache_key] = final_verdicts
    return final_verdicts

# Calculate Draco Score
def calculate_draco_score(rubric_sections: list, judge_verdicts: dict) -> float:
    total_score = 0.0
    max_possible = 0.0
    for s in rubric_sections:
        for c in s.get("criteria", []):
            w = c["weight"]
            c_id = c["id"]
            if w > 0:
                max_possible += w
            val = judge_verdicts.get(c_id, 0.0)
            total_score += w * val
    total_score = max(0.0, total_score)
    return 0.0 if max_possible == 0.0 else (total_score / max_possible) * 100.0

# Prompt injection helper
def get_injected_prompt(task: Dict[str, Any], inject_context: bool) -> str:
    if inject_context and "context" in task and task["context"]:
        return f"{task['problem']}\n\n### CONTEXT:\n{task['context']}"
    return task['problem']

# Complete Evaluation Loop Runner
async def run_evaluation_suite(playback_mode: bool = True, bootstraps: int = 1000):
    print("=============================================================")
    print(f"🧪 DRACO Evaluation Benchmark Suite - Mode: {'CACHE PLAYBACK' if playback_mode else 'LIVE API RUN'}")
    print("=============================================================")
    
    load_cache()
    
    dataset_path = Path("draco_fallback_dataset.json")
    if not dataset_path.exists():
        print("❌ Error: draco_fallback_dataset.json not found!")
        return None
        
    with open(dataset_path, "r", encoding="utf-8") as f:
        tasks = json.load(f)
    print(f"📂 Loaded {len(tasks)} tasks.")
    
    RUN_CONFIGS = [
        {"name": "Gemma-4-E2B alone", "type": "single", "model": "google/gemma-4-e2b-it", "inject_context": False},
        {"name": "Gemma-4-E2B + Context", "type": "single", "model": "google/gemma-4-e2b-it", "inject_context": True},
        {"name": "--fusion panel", "type": "fusion", "inject_context": False},
        {"name": "Fusion + Context", "type": "fusion", "inject_context": True},
        {"name": "Qwen2.5-7B alone", "type": "single", "model": "Qwen/Qwen2.5-7B-Instruct", "inject_context": False},
        {"name": "GLM-5.2 alone", "type": "single", "model": "zai-org/GLM-5.2", "inject_context": False},
        {"name": "GLM-5.2 + Context", "type": "single", "model": "zai-org/GLM-5.2", "inject_context": True},
        {"name": "gpt-4o alone", "type": "single", "model": "gpt-4o", "inject_context": False},
        {"name": "gpt-5.5 alone", "type": "single", "model": "gpt-5.5", "inject_context": False},
        {"name": "gpt-5.5 + Context", "type": "single", "model": "gpt-5.5", "inject_context": True},
    ]
    
    results_report = {"benchmark_name": "DRACO Evaluation Suite", "configurations": {}}
    score_grid = {t["id"]: {} for t in tasks}
    
    for config in RUN_CONFIGS:
        cfg_name = config["name"]
        print(f"🚀 Running: {cfg_name}...")
        results_report["configurations"][cfg_name] = {
            "tasks": [], "mean_score": 0.0, "total_api_cost": 0.0, "total_infra_cost": 0.0,
            "std_dev": 0.0, "ci_lower": 0.0, "ci_upper": 0.0
        }
        
        tot_score, tot_api, tot_infra = 0.0, 0.0, 0.0
        for idx, item in enumerate(tasks):
            t_id = item["id"]
            rubric = json.loads(item["answer"])
            final_prompt = get_injected_prompt(item, config["inject_context"])
            
            if config["type"] == "single":
                out, api_c, infra_c = await call_single_model(config["model"], final_prompt, playback_mode)
            else:
                out = await rust_fusion_pipeline(final_prompt, playback_mode)
                api_c, infra_c = estimate_fusion_cost(final_prompt, out)
                
            verdicts = judge_response_against_rubric(item["problem"], out, item["answer"], playback_mode)
            score = calculate_draco_score(rubric.get("sections", []), verdicts)
            
            tot_score += score
            tot_api += api_c
            tot_infra += infra_c
            score_grid[t_id][cfg_name] = score
            
            results_report["configurations"][cfg_name]["tasks"].append({
                "task_id": t_id, "domain": item["domain"], "score": score,
                "api_cost": api_c, "infra_cost": infra_c
            })
            
        mean_s = tot_score / len(tasks)
        scores_list = [t["score"] for t in results_report["configurations"][cfg_name]["tasks"]]
        std_d, ci_l, ci_u = compute_bootstrap_ci(scores_list, 0.95, bootstraps)
        
        results_report["configurations"][cfg_name].update({
            "mean_score": mean_s, "total_api_cost": tot_api, "total_infra_cost": tot_infra,
            "std_dev": std_d, "ci_lower": ci_l, "ci_upper": ci_u
        })
        print(f"✨ {cfg_name} Mean: {mean_s:.2f}% (±{std_d:.2f}%, 95% CI: [{ci_l:.1f}%, {ci_u:.1f}%]) | API: ${tot_api:.5f} | Infra: ${tot_infra:.5f}")
        
    # Save results json
    with open("draco_benchmark_results.json", "w", encoding="utf-8") as f:
        json.dump(results_report, f, indent=2)
    print("💾 Saved results to draco_benchmark_results.json")
    return results_report



### 🚀 5. Execute Draco Benchmark
Run this cell to start the evaluation.
- Set `PLAYBACK_MODE = True` for instant evaluation using pre-computed responses (No keys required, completes in ~2 seconds).
- Set `PLAYBACK_MODE = False` to run live queries against HuggingFace Inference API (Requires `HF_TOKEN`) and/or OpenAI/Gemini APIs (Requires `OPENAI_API_KEY`/`GOOGLE_GEMINI_API_KEY`).


In [ ]:
#@title Start Evaluation
PLAYBACK_MODE = True #@param {type:"boolean"}
BOOTSTRAP_REPLICATES = 1000 #@param {type:"integer"}

results_report = asyncio.run(run_evaluation_suite(playback_mode=PLAYBACK_MODE, bootstraps=BOOTSTRAP_REPLICATES))



### 📊 6. Interactive Visualizations
Execute the cells below to render premium, publications-quality plots and charts comparing ModelFusion with single-model baselines.

#### Plot A: Comparative Performance Horizontal Bar Chart
Shows mean scores with 95% Confidence Interval error bars.


In [ ]:
#@title Comparative Performance Horizontal Bar Chart
import matplotlib.pyplot as plt
import numpy as np

# Apply professional styled plotting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, ax = plt.subplots(figsize=(12, 7))

configs = list(results_report["configurations"].keys())
means = [results_report["configurations"][c]["mean_score"] for c in configs]
ci_lowers = [results_report["configurations"][c]["ci_lower"] for c in configs]
ci_uppers = [results_report["configurations"][c]["ci_upper"] for c in configs]

# Calculate asymmetric error bars for 95% CI
yerr = [[means[i] - ci_lowers[i] for i in range(len(configs))],
        [ci_uppers[i] - means[i] for i in range(len(configs))]]

# Sort configs by mean score for presentation
sorted_indices = np.argsort(means)
sorted_configs = [configs[i] for i in sorted_indices]
sorted_means = [means[i] for i in sorted_indices]
sorted_yerr = [[yerr[0][i] for i in sorted_indices], [yerr[1][i] for i in sorted_indices]]

# Curated HSL-tailored palette (Plasma)
colors = plt.cm.plasma(np.linspace(0.25, 0.85, len(configs)))

bars = ax.barh(sorted_configs, sorted_means, xerr=sorted_yerr, 
               color=colors, edgecolor='none', height=0.6,
               capsize=5, error_kw={'ecolor': '#555555', 'lw': 1.5})

ax.set_title("DRACO Evaluation Suite - Configuration Comparison\n(Higher is better, error bars show 95% CI)", fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Mean Score (%)", fontsize=12)
ax.set_xlim(0, 105)
ax.xaxis.grid(True, linestyle='--', alpha=0.6)
ax.yaxis.grid(False)

# Add value labels to bars
for bar in bars:
    width = bar.get_width()
    ax.text(width + 1.5, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', 
            va='center', ha='left', fontsize=10, fontweight='bold', color='#333333')

plt.tight_layout()
plt.show()



#### Plot B: ModelFusion Ablation Trajectory
Isolates the distinct performance gains of ModelFusion's compound layers (Base Model, Context RAG, consensus deliberation panels) vs self-hosting infra costs.


In [ ]:
#@title ModelFusion Ablation Trajectory Plot
fig, ax1 = plt.subplots(figsize=(10, 6))

ablation_stages = [
    "Gemma-4-E2B alone",
    "Gemma-4-E2B + Context",
    "--fusion panel",
    "Fusion + Context"
]
ablation_labels = [
    "1. Single Model\n(No Ctx, No Fusion)",
    "2. Context Only\n(Single + Ctx)",
    "3. Fusion Only\n(No Ctx)",
    "4. Fusion + Context\n(Full System)"
]

ablation_scores = [results_report["configurations"][c]["mean_score"] for c in ablation_stages]
ablation_infra_costs = [results_report["configurations"][c]["total_infra_cost"] for c in ablation_stages]

color = '#5e50cc'
ax1.set_xlabel('Ablation Stage', fontsize=12, labelpad=10)
ax1.set_ylabel('Mean Score (%)', color=color, fontsize=12, fontweight='bold')
line1 = ax1.plot(ablation_labels, ablation_scores, color=color, marker='o', linewidth=2.5, markersize=8, label='Accuracy (%)')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(min(ablation_scores) - 8, max(ablation_scores) + 8)

ax2 = ax1.twinx()  
color = '#10b981'
ax2.set_ylabel('Simulated Infra Cost ($)', color=color, fontsize=12, fontweight='bold')
line2 = ax2.plot(ablation_labels, ablation_infra_costs, color=color, marker='s', linewidth=2.5, linestyle='--', markersize=8, label='Infra Cost ($)')
ax2.tick_params(axis='y', labelcolor=color)

# Add values above markers
for i, txt in enumerate(ablation_scores):
    ax1.annotate(f'{txt:.1f}%', (ablation_labels[i], ablation_scores[i] + 1.2), ha='center', fontweight='bold', color='#333333')
for i, txt in enumerate(ablation_infra_costs):
    ax2.annotate(f'${txt:.5f}', (ablation_labels[i], ablation_infra_costs[i] + 0.000002), ha='center', fontweight='bold', color='#333333')

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

plt.title("ModelFusion Ablation Trajectory: Accuracy vs. Hosting Cost\n(Gemma-4-E2B Baseline)", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()



#### Plot C: Technical Domain Performance Breakdown
Compares selected models and ModelFusion configurations across the technical sub-domains (e.g. Concurrency, Distributed Systems, Cloud Infrastructure, Operating Systems, Cryptography, DL Optimization).


In [ ]:
#@title Technical Domain Performance Chart
import pandas as pd

# Extract distinct domains
dataset_path = Path("draco_fallback_dataset.json")
with open(dataset_path, "r", encoding="utf-8") as f:
    tasks = json.load(f)
domains = list(set([t["domain"] for t in tasks]))
domains.sort()

# Selected configurations to compare
display_subset = ["Gemma-4-E2B alone", "Qwen2.5-7B alone", "GLM-5.2 alone", "gpt-4o alone", "Fusion + Context"]

domain_scores = {cfg: [] for cfg in display_subset}
for d in domains:
    for cfg in display_subset:
        cfg_tasks = results_report["configurations"][cfg]["tasks"]
        scores_in_domain = [t["score"] for t in cfg_tasks if t["domain"] == d]
        domain_scores[cfg].append(np.mean(scores_in_domain) if scores_in_domain else 0.0)

x = np.arange(len(domains))
width = 0.15

fig, ax = plt.subplots(figsize=(14, 8))

# Color palette: HSL-tailored premium color values
palette = ['#e15759', '#76b7b2', '#edc948', '#b07aa1', '#4e79a7']

for i, cfg in enumerate(display_subset):
    ax.bar(x + i*width, domain_scores[cfg], width, label=cfg, color=palette[i])

ax.set_ylabel('Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Performance Comparison across Technical Sub-Domains', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x + width * (len(display_subset)-1)/2)
ax.set_xticklabels(domains, rotation=45, ha='right', fontsize=10, fontweight='bold')
ax.legend(loc='upper right', frameon=True)
ax.set_ylim(0, 115)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.xaxis.grid(False)

plt.tight_layout()
plt.show()



### 📋 7. Summary Table

Execute the cell below to print the final tabular comparison summarizing mean score, standard deviation, bootstrap confidence interval, API cost, and hosting infrastructure cost for all configurations.


In [ ]:
#@title Comparative Summary Report Table
headers = ["Configuration", "Mean Score", "Std Dev", "95% CI", "API Cost", "Infra Cost"]
row_format = "{:<25} | {:<10} | {:<8} | {:<16} | {:<9} | {:<10}"
print("="*88)
print(row_format.format(*headers))
print("="*88)

display_configs = [
    "Gemma-4-E2B alone", "Qwen2.5-7B alone", "GLM-5.2 alone", "GLM-5.2 + Context",
    "gpt-4o alone", "gpt-5.5 alone", "gpt-5.5 + Context", "--fusion panel", "Fusion + Context"
]

for k in display_configs:
    cfg = results_report["configurations"][k]
    ci_str = f"[{cfg['ci_lower']:.1f}%, {cfg['ci_upper']:.1f}%]"
    print(row_format.format(
        k, f"{cfg['mean_score']:.2f}%", f"{cfg['std_dev']:.2f}%", ci_str,
        f"${cfg['total_api_cost']:.4f}", f"${cfg['total_infra_cost']:.5f}"
    ))
print("="*88)

